# MT-1 · Merge OARSI Sub-Grades (per-feature masks)

Attaches the OAI sub-grade composites to the OAI rows of the manifest, matched by
subject and side. Uses the three well-populated features (osteophyte, JSN,
sclerosis) and gives each its **own** mask, so a knee that has joint-space narrowing
but is missing an osteophyte reading still contributes its JSN and sclerosis signal.
This avoids discarding partially-graded knees. Other datasets receive NaN sub-grades
and all-zero masks. Writes to scope3_mt; the original manifest is untouched.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config, importlib; importlib.reload(config)
import pandas as pd, numpy as np, re
from pathlib import Path
PROJECT=Path('/content/drive/MyDrive/Master Thesis')
MT_ROOT=PROJECT/'scope3_mt'; MT_ROOT.mkdir(parents=True, exist_ok=True)
manifest=pd.read_csv(str(config.MANIFEST_CSV))
oarsi=pd.read_csv(str(PROJECT/'oai'/'oai_oarsi_labels.csv'))

SUB=['osteophyte_max','jsn_max','sclerosis_max']
SUB=[c for c in SUB if c in oarsi.columns]
print(f'Manifest: {len(manifest):,} rows')
print(f'OARSI: {len(oarsi):,} knees | using sub-features: {SUB}')
print('Coverage in OARSI table:')
for c in SUB: print(f'  {c}: {oarsi[c].notna().sum():,} non-missing')

Mounted at /content/drive
Manifest: 61,558 rows
OARSI: 8,921 knees | using sub-features: ['osteophyte_max', 'jsn_max', 'sclerosis_max']
Coverage in OARSI table:
  osteophyte_max: 5,507 non-missing
  jsn_max: 8,921 non-missing
  sclerosis_max: 5,506 non-missing


## Merge on subject+side

In [2]:
oai_rows=manifest[manifest['dataset']=='oai'].copy()
def parse_oai_key(fn):
    s=str(fn); sid=re.search(r'(\d{7})', s)
    sid=sid.group(1) if sid else None
    m=re.search(r'_(R|L)_', s.upper()); side=m.group(1) if m else None
    return pd.Series({'subject_id':sid,'side':side})
k=oai_rows['filename'].apply(parse_oai_key)
oai_rows['subject_id']=k['subject_id'].astype(str); oai_rows['side']=k['side']
oarsi['subject_id']=oarsi['subject_id'].astype(str)

oai_m=oai_rows.merge(oarsi[['subject_id','side']+SUB], on=['subject_id','side'], how='left')

any_key = oai_m.merge(oarsi[['subject_id','side']].assign(_k=1), on=['subject_id','side'], how='left')['_k'].notna()
print(f'OAI rows matched to an OARSI knee (any): {any_key.sum():,}/{len(oai_m):,} ({any_key.mean()*100:.1f}%)')
print('Per-feature non-missing on OAI rows:')
for c in SUB: print(f'  {c}: {oai_m[c].notna().sum():,} ({oai_m[c].notna().mean()*100:.1f}%)')

OAI rows matched to an OARSI knee (any): 8,547/8,547 (100.0%)
Per-feature non-missing on OAI rows:
  osteophyte_max: 5,253 (61.5%)
  jsn_max: 8,547 (100.0%)
  sclerosis_max: 5,252 (61.4%)


## Build per-feature masks and assemble

Each sub-feature gets a mask column (1 where present, 0 where missing). The
multi-task trainer multiplies each sub-grade loss by its mask, so missing features
simply contribute no gradient for that knee — without discarding the knee's KL label
or its other present sub-features.

In [4]:
mask_cols=[]
for c in SUB:
    m=c.replace('_max','_mask'); oai_m[m]=oai_m[c].notna().astype(int); mask_cols.append(m)
    oai_m[c]=oai_m[c].fillna(-1)

other=manifest[manifest['dataset']!='oai'].copy()
for c in SUB: other[c]=-1.0
for m in mask_cols: other[m]=0

oai_final=oai_m.drop(columns=['subject_id','side'], errors='ignore')
full=pd.concat([oai_final, other], ignore_index=True, sort=False)
full['has_any_subgrade']=full[mask_cols].max(axis=1).astype(int)
if 'arr_idx' in full.columns: full=full.sort_values('arr_idx').reset_index(drop=True)
out=MT_ROOT/'manifest_mt.csv'; full.to_csv(str(out), index=False)

print(f'Saved -> {out}')
print(f'Total rows: {len(full):,}')
print('Per-feature labelled-knee counts (OAI):')
for c,m in zip(SUB,mask_cols):
    print(f'  {c}: {int(full[m].sum()):,} knees with a label')
print(f'\nKnees with at least one sub-grade: {int(full["has_any_subgrade"].sum()):,}')
print('Distributions (where present):')
for c,m in zip(SUB,mask_cols):
    v=full[full[m]==1][c]
    print(f'  {c}: {v.value_counts().sort_index().to_dict()}')


Saved -> /content/drive/MyDrive/Master Thesis/scope3_mt/manifest_mt.csv
Total rows: 61,558
Per-feature labelled-knee counts (OAI):
  osteophyte_max: 8,547 knees with a label
  jsn_max: 8,547 knees with a label
  sclerosis_max: 8,547 knees with a label

Knees with at least one sub-grade: 8,547
Distributions (where present):
  osteophyte_max: {-1.0: 3294, 0.0: 1475, 1.0: 2633, 2.0: 640, 3.0: 505}
  jsn_max: {0.0: 4953, 1.0: 2138, 2.0: 1177, 3.0: 279}
  sclerosis_max: {-1.0: 3295, 0.0: 2965, 1.0: 1198, 2.0: 931, 3.0: 158}
